# Module 4: Snowflake Model Registry

**Snowpark ML Fundamentals - Week 1 | Lunch & Learn**

---

## Learning Objectives
- Register trained models in Snowflake's native Model Registry
- Version models and attach metadata/metrics
- Load models from the registry and run inference
- Understand model lifecycle management

## Key Concept
> The **Snowflake Model Registry** stores models as first-class Snowflake objects.
> No external MLflow/S3 needed. Models are versioned, governed, and callable
> directly from SQL or Python.

---

In [1]:
%load_ext autoreload
%autoreload 2

## 4.1 Setup: Train a Model First

In [2]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.preprocessing.transformers import build_preprocessing_pipeline
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier

session = create_session(env_path='../.env')

# Quick data prep
NUMERIC_COLS = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
LABEL_COL = 'CHURNED'
FEATURE_COLS = [f"{c}_SCALED" for c in NUMERIC_COLS] + [f"{c}_ENCODED" for c in CATEGORICAL_COLS]

df = create_customer_churn_dataset(session, n_rows=5000)
df_processed, _ = build_preprocessing_pipeline(
    df, NUMERIC_COLS, CATEGORICAL_COLS
)
train_df, test_df = split_data(df_processed)

# Train model
model = train_model(train_df, FEATURE_COLS, LABEL_COL, model_type='xgboost')
preds = predict(model, test_df)
metrics = evaluate_binary_classifier(preds, LABEL_COL, 'PREDICTION')
print("Model metrics:", metrics)

DataFrame.flatten() is deprecated since 0.7.0. Use `DataFrame.join_table_function()` instead.


Model metrics: {'accuracy': 0.819846, 'precision': 0.7102803738317757, 'recall': 0.3275862068965517, 'f1_score': 0.44837758112094395}


## 4.2 Register the Model

In [3]:
from snowpark_fundamentals.registry.model_registry import get_registry, log_model, list_models

# Get the current database/schema from the session
current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')

# Connect to the Model Registry
registry = get_registry(session, current_db, current_schema)
print(f"Connected to registry: {current_db}.{current_schema}")

Connected to registry: MLDS_D.PREDICTIONS


In [ ]:
# Log the model with metrics
# XGBoost models use to_xgboost(), sklearn models use to_sklearn()
native_model = model.to_xgboost()

model_version = log_model(
    registry=registry,
    model=native_model,
    model_name='CHURN_PREDICTOR',
    version_name='V1',
    sample_input=test_df.select(FEATURE_COLS).limit(10),
    metrics=metrics,
)

print(f"Model registered: CHURN_PREDICTOR v1")
print(f"Metrics attached: {metrics}")

Model logged successfully.: 100%|██████████| 6/6 [01:30<00:00, 15.14s/it]                          
Model registered: CHURN_PREDICTOR v1
Metrics attached: {'accuracy': 0.819846, 'precision': 0.7102803738317757, 'recall': 0.3275862068965517, 'f1_score': 0.44837758112094395}


## 4.3 Browse the Registry

In [ ]:
# List all registered models
models = list_models(registry)
print("Registered models:")
print(models)

In [6]:
# Get model details
model_ref = registry.get_model('CHURN_PREDICTOR')
print(f"Model: {model_ref.name}")
print(f"Versions: {model_ref.show_versions()}")

Model: CHURN_PREDICTOR
Versions:                         created_on name                     aliases comment  \
0 2026-02-25 09:29:49.253000-05:00   V1  ["DEFAULT","FIRST","LAST"]    None   

  database_name  schema_name       model_name is_default_version  \
0        MLDS_D  PREDICTIONS  CHURN_PREDICTOR               true   

                               functions  \
0  ["PREDICT_PROBA","EXPLAIN","PREDICT"]   

                                            metadata user_data  \
0  {"metrics": {"accuracy": 0.819846, "precision"...        {}   

                                    model_attributes     size  \
0  {"framework":"xgboost","task":"TABULAR_BINARY_...  7049400   

                                         environment  \
0  {"default":{"python_version":"3.10","cuda_vers...   

                                   runnable_in inference_services  
0  ["WAREHOUSE","SNOWPARK_CONTAINER_SERVICES"]                 []  


## 4.4 Load & Run Inference from Registry

This is the key production pattern: load a registered model and score new data.

In [7]:
from snowpark_fundamentals.registry.model_registry import load_model_and_predict

# Score new data using the registered model
registry_predictions = load_model_and_predict(
    registry=registry,
    model_name='CHURN_PREDICTOR',
    version_name='V1',
    input_df=test_df.select(FEATURE_COLS),
)

registry_predictions.show(10)

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"output_feature_0"  |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|-1.4536816815  |0.523941485064372       |0.5577318136234114        |0.8146184680621967      |-0.3151768044             |1.4348930167                   |0.3059337042       

## 4.5 SQL Access (Bonus)

Registered models can also be called from SQL:

In [8]:
# Show the SQL-callable model reference
print("Models are also accessible via SQL:")
print(f"  SELECT {current_db}.{current_schema}.CHURN_PREDICTOR!PREDICT(...)")
print("  This enables any SQL user to use the model without Python!")

Models are also accessible via SQL:
  SELECT MLDS_D.PREDICTIONS.CHURN_PREDICTOR!PREDICT(...)
  This enables any SQL user to use the model without Python!


## 4.6 Cleanup

In [9]:
from snowpark_fundamentals.registry.model_registry import delete_model

# Clean up the tutorial model (optional)
# Uncomment to delete:
# delete_model(registry, 'CHURN_PREDICTOR')
# print("Model deleted.")

## Key Takeaways

1. **Snowflake Model Registry** is a native, governed store for ML models
2. Supports **scikit-learn, XGBoost, LightGBM, PyTorch, TensorFlow, HuggingFace**
3. Models are **versioned** with attached metrics and metadata
4. Inference runs **inside Snowflake** - no external serving needed
5. Models are callable from both **Python and SQL**

---

**Next: [05 - End-to-End Pipeline](05_end_to_end_pipeline.ipynb)**

In [10]:
session.close()